# Red ER como ley universal: ajuste global y prueba de sobreajuste

**Proyecto:** *Vacuum Particle Journey*, segunda parte.

En el notebook anterior, la red ER ajustaba bien NGC 3198, pero con **3 parámetros libres por galaxia** ($\Upsilon_\star, \kappa, \lambda$). Un modelo tan flexible puede ajustar casi cualquier curva: eso es **sobreajuste** (*overfitting*). El modelo "aprende" el ruido de cada galaxia en lugar de una ley física.

Si los puentes ER son física de verdad, **$\kappa$ (cuánta energía aporta el entrelazamiento) y $\lambda$ (su alcance) deberían ser los mismos en todo el universo**. Aquí lo ponemos a prueba:

1. **Ajuste global:** un único $(\kappa, \lambda)$ para todas las galaxias. En cada galaxia solo queda libre $\Upsilon_\star$, la masa de sus estrellas, que también es libre en todos los demás modelos.
2. **Cuatro pruebas de sobreajuste:**
   - **BIC total**: ¿compensan los $2N$ parámetros extra del modelo libre?
   - **Dispersión de $\kappa, \lambda$** galaxia a galaxia: ¿son compatibles con un solo valor? ¿Dependen de la masa?
   - **Validación cruzada entre galaxias**: aprender $(\kappa, \lambda)$ con el 80 % de las galaxias y predecir el 20 % restante, que el modelo no ha visto.
   - **Extrapolación radial**: ajustar solo el 60 % interior de cada curva y predecir el 40 % exterior. Un modelo sobreajustado falla justo ahí.
3. **Detalle de tres galaxias:** NGC 3198 (espiral grande), NGC 2403 (espiral mediana) y DDO 154 (enana dominada por gas, el caso más "oscuro").

**Comparamos con:**
- NFW libre (3 parámetros);
- **NFW ΛCDM**, con la concentración $c$ fijada por la relación $c$–$M$ de las simulaciones (Dutton y Macciò 2014), es decir 2 parámetros;
- **Verlinde**, que ahora calcula la derivada sobre un perfil de masa suavizado (en el notebook anterior el ruido producía los dientes de sierra).

> Tiempo en Colab: ~10–15 min con las 175 galaxias. Pon `QUICK = True` para una prueba rápida con 40 galaxias.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.integrate import cumulative_trapezoid
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False          # True = 40 galaxias, para probar rápido

# Un color fijo por modelo en todas las figuras
COL = {
    "Datos": "#222222",
    "Solo bariones": "#8a8a85",
    "NFW libre": "#2a78d6",
    "NFW ΛCDM (c–M)": "#6250d6",
    "Verlinde": "#eb6834",
    "Red ER libre": "#eda100",
    "Red ER global": "#1baf7a",
}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6            # kpc (km/s)^2 / Msun
C_KMS = 299792.458
H0 = 70.0                 # km/s/Mpc
H0_KPC = H0 / 1000.0      # km/s/kpc
A0 = C_KMS * H0_KPC       # a0 = c*H0 en (km/s)^2/kpc

## 1. Datos SPARC y control de calidad

Además de las curvas, intentamos bajar la tabla general de SPARC para aplicar el corte de calidad habitual: quitar las curvas de calidad $Q=3$ y las galaxias casi de cara (inclinación < 30°), cuya velocidad es poco fiable. Si la tabla no se descarga, se usan todas las galaxias.

Si no hay forma de obtener los datos, el notebook genera **galaxias sintéticas** para que el código se pueda ejecutar. Sus resultados **no tienen valor físico**.

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    # Tabla 1 de SPARC: nombre, T, D, e_D, f_D, Inc, e_Inc, L, e_L, Reff, SBeff, Rdisk, SBdisk, MHI, RHI, Vflat, e_Vflat, Q, Ref
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok]
    o = np.argsort(g["r"])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

# ---------- galaxias sintéticas (solo si no hay datos reales) ----------
def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    names = ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]
    for name in names:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        d = np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r]
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        os.makedirs("synthetic", exist_ok=True); np.savetxt(path, d, header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: se usan galaxias SINTÉTICAS (con halo NFW). Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = {"NGC3198", "NGC2403", "DDO154"}
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in list(keep & set(GAL)) + list(RNG.choice(rest, 37, replace=False))}
print(f"Galaxias en la muestra: {len(GAL)} | puntos totales: {sum(len(g['r']) for g in GAL.values())}")

## 2. Modelos

En todos, $v_b^2 = v_{gas}^2 + \Upsilon_\star\, v_{\star}^2$ (con el bulbo a $1.4\,\Upsilon_\star$), y $\Upsilon_\star$ tiene un *prior* log-normal centrado en 0.5 con 0.1 dex.

- **Red ER:** $\;v^2 = v_b^2 + \kappa\,\mathcal{K}_\lambda[v_b^2]$. El operador $\mathcal{K}_\lambda$ reparte la masa bariónica con el núcleo $e^{-d/\lambda}/(4\pi\lambda^2 d)$, como en el notebook anterior. Es **lineal** en $v_b^2$, así que para cada $\lambda$ basta con calcularlo una vez por galaxia: $\mathcal{K}_\lambda[v_{gas}^2]$ y $\mathcal{K}_\lambda[v_\star^2]$. Eso hace el ajuste global rápido.
- **Verlinde:** $v_D^2 = \sqrt{\tfrac{a_0}{6}\tfrac{d(r^2 v_b^2)}{dr}}$, ahora con $M_b(r)$ suavizada (0.08 dex en $\log r$) antes de derivar.
- **NFW ΛCDM:** $\log_{10} c = 0.905 - 0.101\,\log_{10}\!\big(M_{200}/10^{12}h^{-1}M_\odot\big)$, con $M_{200} = V_{200}^3/(10\,G H_0)$.

In [ ]:
def er_unit(r, v2, lam, n_grid=150):
    # v²_ER por unidad de κ (lineal en v2), con G·M_b(r) = r·v_b²
    GM = r * v2
    rg = np.linspace(r.max() / n_grid, r.max(), n_grid)
    GMg = np.interp(rg, np.r_[0.0, r], np.r_[0.0, GM])
    dGM = np.diff(np.r_[0.0, GMg])
    S, R = rg[:, None], rg[None, :]
    kern = (np.exp(-np.abs(S - R) / lam) - np.exp(-(S + R) / lam)) / (8 * np.pi * lam * S * R)
    GM_er = cumulative_trapezoid(4 * np.pi * rg**2 * (kern @ dGM), rg, initial=0.0)
    return np.interp(r, rg, GM_er) / r

def er_parts(g, lam):
    return er_unit(g["r"], g["vgas2"], lam), er_unit(g["r"], g["vstar2"], lam)

def v2_verlinde_extra(r, vb2, smooth_dex=0.08):
    GM = np.maximum.accumulate(r * np.clip(vb2, 0, None))
    lr = np.log10(r)
    grid = np.linspace(lr.min(), lr.max(), 200)
    GMg = np.interp(grid, lr, GM)
    dx = grid[1] - grid[0] if len(grid) > 1 else 1.0
    n = max(int(3 * smooth_dex / dx), 1)
    k = np.exp(-0.5 * (np.arange(-n, n + 1) * dx / smooth_dex) ** 2); k /= k.sum()
    GMs = np.convolve(np.pad(GMg, n, mode="edge"), k, mode="valid")
    rgrid = 10 ** grid
    dy = np.gradient(rgrid * GMs, rgrid)
    return np.interp(r, rgrid, np.sqrt(A0 / 6 * np.clip(dy, 0, None)))

def c_dutton(V200):
    M200 = V200**3 / (10 * G * H0_KPC)
    return 10 ** (0.905 - 0.101 * np.log10(M200 / (1e12 / (H0 / 100))))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

MODELS = {
    "Solo bariones":  dict(names=["Υ"], p0=[0.5], lo=[0.1], hi=[1.5],
                           v2=lambda g, p: vb2(g, p[0])),
    "NFW libre":      dict(names=["Υ", "V200", "c"], p0=[0.5, 100, 10], lo=[0.1, 10, 1], hi=[1.5, 500, 50],
                           v2=lambda g, p: vb2(g, p[0]) + _v2_nfw(g["r"], p[1], p[2])),
    "NFW ΛCDM (c–M)": dict(names=["Υ", "V200"], p0=[0.5, 100], lo=[0.1, 10], hi=[1.5, 500],
                           v2=lambda g, p: vb2(g, p[0]) + _v2_nfw(g["r"], p[1], c_dutton(p[1]))),
    "Verlinde":       dict(names=["Υ"], p0=[0.5], lo=[0.1], hi=[1.5],
                           v2=lambda g, p: vb2(g, p[0]) + v2_verlinde_extra(g["r"], vb2(g, p[0]))),
    "Red ER libre":   dict(names=["Υ", "κ", "λ"], p0=[0.5, 3, 10], lo=[0.1, 0, 0.3], hi=[1.5, 300, 300],
                           v2=lambda g, p: vb2(g, p[0]) + p[1] * er_unit(g["r"], vb2(g, p[0]), p[2])),
}

## 3. Herramientas de ajuste

`mask` permite ajustar solo una parte de los puntos (para la prueba de extrapolación) y medir el $\chi^2$ en otra parte.

In [ ]:
def prior(u):
    return np.log10(u / 0.5) / 0.1

def chi2_on(g, v, mask):
    return float(np.sum(((g["vobs"] - v) / g["err"])[mask] ** 2))

def fit_model(g, name, mask=None, n_starts=4, rng=RNG):
    m = MODELS[name]
    N = len(g["r"]); mask = np.ones(N, bool) if mask is None else mask
    lo, hi = np.array(m["lo"], float), np.array(m["hi"], float)
    def res(p):
        v = np.sqrt(np.clip(m["v2"](g, p), 0, None))
        return np.r_[((g["vobs"] - v) / g["err"])[mask], prior(p[0])]
    starts = [np.clip(np.array(m["p0"], float), lo, hi)] + [lo + rng.random(len(lo)) * (hi - lo) for _ in range(n_starts - 1)]
    best = None
    for p0 in starts:
        try:
            r = least_squares(res, p0, bounds=(lo, hi), method="trf")
            if best is None or r.cost < best.cost:
                best = r
        except Exception:
            pass
    v = np.sqrt(np.clip(m["v2"](g, best.x), 0, None))
    chi2 = chi2_on(g, v, mask); n = int(mask.sum()); k = len(best.x)
    try:
        perr = np.sqrt(np.diag(np.linalg.pinv(best.jac.T @ best.jac)) * max(chi2 / max(n - k, 1), 1))
    except Exception:
        perr = np.full(k, np.nan)
    return dict(p=best.x, perr=perr, v=v, chi2=chi2, n=n, k=k)

def fit_ups_only(g, A, B, mask=None):
    # v² = A + Υ·B ; solo Υ libre (modelo ER con κ, λ fijos)
    mask = np.ones(len(g["r"]), bool) if mask is None else mask
    def f(lu):
        u = 10 ** lu
        v = np.sqrt(np.clip(A + u * B, 0, None))
        return chi2_on(g, v, mask) + prior(u) ** 2
    r = minimize_scalar(f, bounds=(-1, np.log10(1.5)), method="bounded")
    u = 10 ** r.x
    v = np.sqrt(np.clip(A + u * B, 0, None))
    return dict(p=np.array([u]), v=v, chi2=chi2_on(g, v, mask), n=int(mask.sum()), k=1)

def fit_er_global_one(g, kappa, lam, mask=None, parts=None):
    Eg, Es = er_parts(g, lam) if parts is None else parts
    return fit_ups_only(g, g["vgas2"] + kappa * Eg, g["vstar2"] + kappa * Es, mask)

## 4. Ajuste global: un solo $(\kappa, \lambda)$ para todo el universo

Buscamos el mínimo de $\chi^2_{total}(\kappa,\lambda) = \sum_{galaxias}\min_{\Upsilon}\chi^2_i$ en dos pasadas: una rejilla gruesa y luego una fina alrededor del mínimo. Para cada $(\kappa, \lambda)$ se optimiza $\Upsilon$ en cada galaxia.

**Incertidumbre:** la región $\Delta\chi^2 \le 2.30\,s$ es el intervalo de 1σ para 2 parámetros. El factor $s = \chi^2_{min}/\text{g.l.}$ infla el error si el ajuste global no es perfecto.

In [ ]:
U_GRID = np.geomspace(0.1, 1.5, 60)
PRIOR_U = prior(U_GRID) ** 2

def chi2_surface(galaxies, lams, kaps, return_best_u=False):
    surf = np.zeros((len(lams), len(kaps)))
    for i, lam in enumerate(lams):
        for g in galaxies.values():
            Eg, Es = er_parts(g, lam)
            A = g["vgas2"][None, :] + kaps[:, None] * Eg[None, :]
            B = g["vstar2"][None, :] + kaps[:, None] * Es[None, :]
            v = np.sqrt(np.clip(A[:, None, :] + U_GRID[None, :, None] * B[:, None, :], 0, None))
            c = np.sum(((g["vobs"] - v) / g["err"]) ** 2, axis=2) + PRIOR_U[None, :]
            surf[i] += c.min(axis=1)
    return surf

def global_fit(galaxies, verbose=True):
    t0 = time.time()
    lams = np.geomspace(0.3, 300, 24); kaps = np.r_[0.0, np.geomspace(0.01, 300, 70)]
    s1 = chi2_surface(galaxies, lams, kaps)
    i, j = np.unravel_index(np.argmin(s1), s1.shape)
    if verbose and (i in (0, len(lams) - 1) or j in (1, len(kaps) - 1)):
        print("Aviso: el mínimo global está en el borde de la rejilla "
              f"(λ = {lams[i]:.3g} kpc, κ = {kaps[j]:.3g}). "
              + ("λ mínimo significa que la red ER solo reescala la masa visible, sin halo extendido."
                 if i == 0 else ""))
    lam_c, kap_c = lams[i], max(kaps[j], 0.01)
    lams2 = np.unique(np.clip(lam_c * np.geomspace(10 ** -0.2, 10 ** 0.2, 13), 0.3, 300))
    kaps2 = kap_c * np.geomspace(10 ** -0.3, 10 ** 0.3, 41)
    s2 = chi2_surface(galaxies, lams2, kaps2)
    i2, j2 = np.unravel_index(np.argmin(s2), s2.shape)
    out = dict(lam=lams2[i2], kappa=kaps2[j2], chi2=s2[i2, j2], coarse=(lams, kaps, s1), fine=(lams2, kaps2, s2))
    if verbose:
        print(f"Ajuste global en {time.time()-t0:.0f} s")
    return out

GF = global_fit(GAL)
N_tot = sum(len(g["r"]) for g in GAL.values())
K_glob = len(GAL) + 2
s_fac = max(GF["chi2"] / (N_tot - K_glob), 1.0)
lams2, kaps2, s2 = GF["fine"]
inside = (s2 - s2.min()) <= 2.30 * s_fac
L, Kp = np.meshgrid(lams2, kaps2, indexing="ij")
print(f"κ global = {GF['kappa']:.3g}   (1σ: {Kp[inside].min():.3g} – {Kp[inside].max():.3g})")
print(f"λ global = {GF['lam']:.3g} kpc (1σ: {L[inside].min():.3g} – {L[inside].max():.3g} kpc)")
print(f"χ²/g.l. global = {GF['chi2']/(N_tot-K_glob):.2f}  (factor s = {s_fac:.2f})")
if inside.any() and (L[inside].max() >= lams2.max() or Kp[inside].max() >= kaps2.max()):
    print("Aviso: la región de 1σ toca el borde de la rejilla fina; el error real es mayor.")

## 5. Ajuste galaxia a galaxia de todos los modelos

Ajustamos cada modelo por separado en cada galaxia (es lo que tarda más) y añadimos la red ER global con el $(\kappa,\lambda)$ que acabamos de encontrar.

In [ ]:
t0 = time.time()
rows, FITS = [], {}
for n_i, (name, g) in enumerate(GAL.items()):
    FITS[name] = {m: fit_model(g, m) for m in MODELS}
    FITS[name]["Red ER global"] = fit_er_global_one(g, GF["kappa"], GF["lam"])
    for m, f in FITS[name].items():
        rows.append(dict(galaxia=name, modelo=m, chi2=f["chi2"], n=f["n"], k=f["k"],
                         chi2r=f["chi2"] / max(f["n"] - f["k"], 1), **{f"p{j}": v for j, v in enumerate(f["p"])}))
    if (n_i + 1) % 25 == 0:
        print(f"  {n_i+1}/{len(GAL)} galaxias, {time.time()-t0:.0f} s")
RES = pd.DataFrame(rows)
ORDER = ["Solo bariones", "NFW libre", "NFW ΛCDM (c–M)", "Verlinde", "Red ER libre", "Red ER global"]
print(f"Hecho en {time.time()-t0:.0f} s")

## 6. Prueba 1: BIC total de la muestra

$\text{BIC}_{total} = \chi^2_{total} + K_{total}\ln N_{total}$. El modelo libre de la red ER gasta $3N_{gal}$ parámetros y el global $N_{gal}+2$. Si el libre no baja el $\chi^2$ lo suficiente para compensar, **sus parámetros extra eran sobreajuste**. Una diferencia $\Delta\text{BIC} > 10$ es evidencia muy fuerte.

In [ ]:
summ = []
for m in ORDER:
    d = RES[RES.modelo == m]
    K = d.k.sum() + (2 if m == "Red ER global" else 0)
    chi2 = d.chi2.sum()
    summ.append(dict(Modelo=m, parametros_totales=int(K), chi2_total=round(chi2, 0),
                     chi2r_mediana=round(d.chi2r.median(), 2),
                     galaxias_chi2r_menor_2=f"{np.mean(d.chi2r < 2):.0%}",
                     BIC_total=round(chi2 + K * np.log(N_tot), 0)))
SUMM = pd.DataFrame(summ).set_index("Modelo")
SUMM["ΔBIC"] = SUMM.BIC_total - SUMM.BIC_total.min()
SUMM

## 7. Prueba 2: ¿son $\kappa$ y $\lambda$ iguales en todas las galaxias?

A la izquierda, el mapa de $\Delta\chi^2$ del ajuste global (cuanto más oscuro, mejor). A la derecha, los $(\kappa, \lambda)$ que elige cada galaxia cuando se ajusta por libre; la estrella es el valor global. Si la red ER fuera universal, los puntos se agruparían en torno a la estrella.

Debajo: ¿depende $\kappa$ de la masa bariónica de la galaxia? Una correlación fuerte significaría que $\kappa$ no es una constante, pero podría ser una **ley de escala** $\kappa \propto M_b^\beta$, que sería una pista física interesante.

In [ ]:
lams, kaps, s1 = GF["coarse"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.3))
kk = kaps[1:]
d = np.clip(s1[:, 1:] - min(s1.min(), GF["chi2"]), 0, None) / s_fac
pm = a1.pcolormesh(np.log10(kk), np.log10(lams), np.log10(1 + d), cmap="Greens_r", shading="auto")
a1.contour(np.log10(kk), np.log10(lams), d, levels=[11.8], colors="#222222", linewidths=1)
a1.plot(np.log10(GF["kappa"]), np.log10(GF["lam"]), "*", ms=14, color="#222222")
a1.set_xlabel("log10 κ"); a1.set_ylabel("log10 λ [kpc]")
a1.set_title("Ajuste global (línea: región 3σ)", loc="left")
fig.colorbar(pm, ax=a1, label="log10(1 + Δχ²/s)   (oscuro = mejor)")

er = RES[RES.modelo == "Red ER libre"]
a2.plot(np.log10(er.p1.clip(lower=1e-2)), np.log10(er.p2), "o", ms=5, color=COL["Red ER libre"], alpha=0.75,
        label="Cada galaxia (ajuste libre)")
a2.plot(np.log10(GF["kappa"]), np.log10(GF["lam"]), "*", ms=16, color=COL["Red ER global"], mec="#222222",
        label="Valor global")
a2.set_xlabel("log10 κ"); a2.set_ylabel("log10 λ [kpc]")
a2.set_title("Ajuste libre: cada galaxia elige su (κ, λ)", loc="left"); a2.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("Dispersión del ajuste libre:")
print(f"  log10 κ: mediana {np.log10(er.p1.clip(lower=1e-2)).median():.2f}, dispersión (16–84 %) "
      f"{np.subtract(*np.percentile(np.log10(er.p1.clip(lower=1e-2)), [84, 16]))/2:.2f} dex")
print(f"  log10 λ: mediana {np.log10(er.p2).median():.2f}, dispersión (16–84 %) "
      f"{np.subtract(*np.percentile(np.log10(er.p2), [84, 16]))/2:.2f} dex")
print(f"  galaxias en el borde de λ (≥290 kpc): {np.mean(er.p2 >= 290):.0%}, κ≈0: {np.mean(er.p1 < 0.02):.0%}")

In [ ]:
Mb, kap = [], []
for name, g in GAL.items():
    u = FITS[name]["Red ER libre"]["p"][0]
    Mb.append(max(g["r"][-1] * vb2(g, u)[-1] / G, 1e5))
    kap.append(FITS[name]["Red ER libre"]["p"][1])
Mb, kap = np.array(Mb), np.array(kap)
ok = kap > 0.02
rho, pval = spearmanr(np.log10(Mb[ok]), np.log10(kap[ok]))
beta, b0 = np.polyfit(np.log10(Mb[ok]), np.log10(kap[ok]), 1)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.loglog(Mb[ok], kap[ok], "o", ms=5, color=COL["Red ER libre"], alpha=0.75, label="Ajuste libre por galaxia")
mm = np.geomspace(Mb[ok].min(), Mb[ok].max(), 50)
ax.loglog(mm, 10 ** b0 * mm ** beta, color="#222222", lw=1.5, label=f"Ajuste: κ ∝ M_b^{beta:.2f}")
ax.axhline(GF["kappa"], color=COL["Red ER global"], ls="--", label="κ global")
ax.set_xlabel("Masa bariónica [M☉]"); ax.set_ylabel("κ")
ax.set_title("¿Depende κ de la masa de la galaxia?", loc="left"); ax.legend(fontsize=9)
plt.show()
print(f"Correlación de Spearman log κ – log M_b: ρ = {rho:.2f} (p = {pval:.1e}); pendiente β = {beta:.2f}")

## 8. Prueba 3: validación cruzada entre galaxias

Dividimos las galaxias en 5 grupos. Para cada grupo:
1. Ajustamos $(\kappa,\lambda)$ **solo con las otras galaxias**.
2. Aplicamos esos valores al grupo que el modelo no ha visto (solo se ajusta $\Upsilon$) y medimos $\chi^2_\nu$.

Si el $\chi^2_\nu$ fuera de muestra ≈ el $\chi^2_\nu$ dentro de muestra, el modelo **generaliza**: es una ley, no un ajuste a medida. Guardamos también el $(\kappa,\lambda)$ de cada grupo para usarlo en la prueba 4 sin hacer trampa.

In [ ]:
names = np.array(list(GAL))
perm = RNG.permutation(len(names))
folds = np.array_split(perm, 5)
CV_PARAMS, cv_rows = {}, []
t0 = time.time()
for fi, test_idx in enumerate(folds):
    test = set(names[test_idx])
    train = {k: v for k, v in GAL.items() if k not in test}
    gf = global_fit(train, verbose=False)
    for name in test:
        CV_PARAMS[name] = (gf["kappa"], gf["lam"])
        f = fit_er_global_one(GAL[name], gf["kappa"], gf["lam"])
        f_in = FITS[name]["Red ER global"]
        cv_rows.append(dict(grupo=fi, galaxia=name, kappa=gf["kappa"], lam=gf["lam"],
                            chi2r_fuera=f["chi2"] / max(f["n"] - 1, 1),
                            chi2r_dentro=f_in["chi2"] / max(f_in["n"] - 1, 1)))
    print(f"  grupo {fi+1}/5: κ = {gf['kappa']:.3g}, λ = {gf['lam']:.3g} kpc   ({time.time()-t0:.0f} s)")
CV = pd.DataFrame(cv_rows)
ratio_cv = CV.chi2r_fuera.median() / CV.chi2r_dentro.median()
print(f"\nχ²ν mediana dentro de muestra: {CV.chi2r_dentro.median():.2f} | fuera de muestra: {CV.chi2r_fuera.median():.2f}"
      f" | cociente: {ratio_cv:.2f}")
print(f"Variación de κ entre grupos: {CV.groupby('grupo').kappa.first().std()/GF['kappa']:.0%} | "
      f"de λ: {CV.groupby('grupo').lam.first().std()/GF['lam']:.0%}")

## 9. Prueba 4: extrapolación radial (la más dura)

Para cada galaxia con 10 puntos o más, ajustamos solo el **60 % interior** de la curva y predecimos el **40 % exterior**, que es justo donde "aparece" la materia oscura. La masa bariónica es conocida en todos los radios (gas y estrellas se observan); solo ocultamos $V_{obs}$.

- **NFW libre** y **Red ER libre** tienen libertad para ajustar el interior a medida.
- **Red ER global** usa el $(\kappa,\lambda)$ de la validación cruzada, aprendido **sin esa galaxia**.
- **Verlinde** y **NFW ΛCDM** tienen muy poca libertad.

Un modelo sobreajustado tiene un buen $\chi^2$ dentro y uno malo fuera.

In [ ]:
ext_rows = []
t0 = time.time()
for name, g in GAL.items():
    N = len(g["r"])
    if N < 10:
        continue
    cut = int(np.floor(0.6 * N))
    tr = np.zeros(N, bool); tr[:cut] = True
    te = ~tr
    for m in ["NFW libre", "NFW ΛCDM (c–M)", "Verlinde", "Red ER libre"]:
        f = fit_model(g, m, mask=tr, n_starts=3)
        ext_rows.append(dict(galaxia=name, modelo=m, chi2r_dentro=f["chi2"] / max(tr.sum() - f["k"], 1),
                             chi2r_fuera=chi2_on(g, f["v"], te) / te.sum()))
    kap_cv, lam_cv = CV_PARAMS[name]
    f = fit_er_global_one(g, kap_cv, lam_cv, mask=tr)
    ext_rows.append(dict(galaxia=name, modelo="Red ER global", chi2r_dentro=f["chi2"] / max(tr.sum() - 1, 1),
                         chi2r_fuera=chi2_on(g, f["v"], te) / te.sum()))
EXT = pd.DataFrame(ext_rows)
print(f"Hecho en {time.time()-t0:.0f} s, {EXT.galaxia.nunique()} galaxias")
ext_sum = EXT.groupby("modelo").agg(chi2r_interior=("chi2r_dentro", "median"),
                                    chi2r_exterior=("chi2r_fuera", "median"))
ext_sum["empeora_x"] = ext_sum.chi2r_exterior / ext_sum.chi2r_interior
ext_sum.loc[[m for m in ORDER if m in ext_sum.index]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bins = np.linspace(-1.5, 3, 37)
for m in ["NFW libre", "NFW ΛCDM (c–M)", "Verlinde", "Red ER libre", "Red ER global"]:
    x = np.log10(EXT.loc[EXT.modelo == m, "chi2r_fuera"].clip(lower=10**-1.5))
    ax.hist(x, bins=bins, histtype="step", lw=2, color=COL[m], label=m)
ax.axvline(0, color="#c3c2b7", lw=1)
ax.set_xlabel("log10 χ²ν en el 40 % exterior (predicho, no ajustado)")
ax.set_ylabel("Número de galaxias")
ax.set_title("Prueba de extrapolación radial", loc="left"); ax.legend(fontsize=9)
plt.show()

## 10. Tres galaxias en detalle: NGC 3198, NGC 2403 y DDO 154

- **NGC 3198:** espiral grande con una curva plana muy limpia hasta ~45 kpc.
- **NGC 2403:** espiral mediana, con muchos puntos y errores pequeños: muy exigente.
- **DDO 154:** enana dominada por gas, donde casi toda la masa es "oscura". Es la prueba más dura para cualquier alternativa a la materia oscura.

La línea vertical punteada marca dónde termina el 60 % interior de la prueba de extrapolación.

In [ ]:
def plot_detail(name):
    if name not in GAL:
        print(f"{name} no está en la muestra (¿corte de calidad?)"); return
    g, F = GAL[name], FITS[name]
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(8.5, 6.8), sharex=True,
                                  gridspec_kw=dict(height_ratios=[3, 1.2], hspace=0.06))
    r = g["r"]
    ax.errorbar(r, g["vobs"], g["err"], fmt="o", ms=4, color=COL["Datos"], ecolor="#9a9a95", elinewidth=1,
                label="Observado", zorder=5)
    for m in ORDER:
        f = F[m]; chi2r = f["chi2"] / max(f["n"] - f["k"], 1)
        ls = "--" if m in ("Solo bariones", "Red ER libre") else "-"
        ax.plot(r, f["v"], ls, color=COL[m], label=f"{m} (χ²ν={chi2r:.1f})")
        if m != "Solo bariones":
            axr.plot(r, (g["vobs"] - f["v"]) / g["err"], "o-", ms=2.5, lw=1, color=COL[m])
    if len(r) >= 10:
        for a in (ax, axr):
            a.axvline(r[int(np.floor(0.6 * len(r))) - 1], color="#9a9a95", ls=":", lw=1.2)
    ax.set_ylim(0, None); ax.set_ylabel("Velocidad [km/s]")
    ax.set_title(f"{name}", loc="left"); ax.legend(fontsize=8.5, loc="lower right")
    axr.axhline(0, color="#c3c2b7", lw=1); axr.set_ylabel("Residuo [σ]"); axr.set_xlabel("Radio [kpc]")
    plt.show()
    t = pd.DataFrame([dict(Modelo=m, k=F[m]["k"], chi2r=round(F[m]["chi2"] / max(F[m]["n"] - F[m]["k"], 1), 2),
                           Parametros=", ".join(f"{a}={b:.3g}" for a, b in zip(
                               MODELS[m]["names"] if m in MODELS else ["Υ"], F[m]["p"])))
                      for m in ORDER]).set_index("Modelo")
    e = EXT[EXT.galaxia == name].set_index("modelo")[["chi2r_dentro", "chi2r_fuera"]].round(2)
    e.columns = ["extrap_chi2r_interior", "extrap_chi2r_exterior"]
    return t.join(e)

plot_detail("NGC3198")

In [ ]:
plot_detail("NGC2403")

In [ ]:
plot_detail("DDO154")

## 11. Veredicto automático sobre el sobreajuste

La celda siguiente resume las cuatro pruebas con reglas fijas, para que la conclusión no dependa de lo que queramos ver.

In [ ]:
def veredicto():
    msgs = []
    if not HAVE_SPARC:
        msgs.append("⚠️  Datos SINTÉTICOS (generados con NFW): este veredicto solo comprueba que el código funciona.")
    b_free, b_glob = SUMM.loc["Red ER libre", "BIC_total"], SUMM.loc["Red ER global", "BIC_total"]
    d = b_free - b_glob
    if d > 10:
        msgs.append(f"1) BIC: el modelo GLOBAL es preferido (ΔBIC = {d:.0f}). Los κ y λ por galaxia eran sobreajuste.")
    elif d < -10:
        msgs.append(f"1) BIC: los datos piden κ y λ distintos por galaxia (ΔBIC = {-d:.0f} a favor del libre). "
                    "Con constantes universales la red ER no basta tal como está planteada.")
    else:
        msgs.append(f"1) BIC: no concluyente (|ΔBIC| = {abs(d):.0f} < 10).")
    best = SUMM.BIC_total.idxmin()
    msgs.append(f"   El mejor modelo por BIC total es: {best}. "
                f"Red ER global frente a NFW ΛCDM: ΔBIC = {b_glob - SUMM.loc['NFW ΛCDM (c–M)', 'BIC_total']:+.0f} "
                f"(negativo = gana la red ER).")
    msgs.append(f"2) Dispersión: κ por galaxia varía {np.subtract(*np.percentile(np.log10(kap[ok]), [84, 16]))/2:.2f} dex; "
                f"correlación con la masa ρ = {rho:.2f} (p = {pval:.0e}). "
                + ("κ depende de la masa: probar una ley κ ∝ M_b^β." if (abs(rho) > 0.4 and pval < 0.01)
                   else "Sin dependencia clara con la masa."))
    msgs.append(f"3) Validación cruzada: fuera/dentro = {ratio_cv:.2f} → "
                + ("generaliza bien (no hay sobreajuste entre galaxias)." if ratio_cv < 1.25
                   else "empeora fuera de muestra: hay sobreajuste."))
    e = ext_sum["empeora_x"]
    msgs.append(f"4) Extrapolación: χ²ν exterior/interior = Red ER libre {e.get('Red ER libre', np.nan):.1f}x, "
                f"Red ER global {e.get('Red ER global', np.nan):.1f}x, NFW libre {e.get('NFW libre', np.nan):.1f}x, "
                f"Verlinde {e.get('Verlinde', np.nan):.1f}x.")
    ex = ext_sum["chi2r_exterior"]
    msgs.append(f"   Mejor predicción del exterior: {ex.idxmin()} (χ²ν = {ex.min():.2f}).")
    print("\n".join(msgs))

veredicto()

## 12. Cómo seguir

- **Si el modelo global aguanta** (BIC y validación cruzada favorables): el siguiente paso es *predecir* $\kappa$ y $\lambda$ desde la física. Por ejemplo, comprobar si $\lambda$ se acerca a alguna escala natural ($c/H_0 \approx 4.3$ Gpc es demasiado grande; $\sqrt{GM_b/a_0}$, el "radio MOND", depende de la galaxia) o si $\kappa$ sale de la densidad de entrelazamiento generada en la época del plasma de quarks y gluones.
- **Si $\kappa$ depende de la masa**: añadir $\kappa = \kappa_0 (M_b/10^{10}M_\odot)^\beta$ como modelo global de 3 parámetros. Un $\beta \approx -1/2$ sería muy sugerente, porque es la escala que aparece en Verlinde y en MOND ($v^4 \propto M_b$).
- **Si falla en DDO 154 y en las enanas**: es el punto débil típico de las alternativas a la materia oscura. Habría que revisar si el núcleo $K$ necesita otra forma (por ejemplo $1/d^2$, sin escala) en lugar de un exponencial.
- **Más allá de las curvas de rotación**: lentes gravitacionales débiles (Brouwer et al. 2017 probaron Verlinde así) y el Cúmulo Bala.